# 03 – Main corpus: download, split, paragraph-number (M01–M30, R01–R05)
Run top to bottom. Each Gutenberg book is downloaded once (CLAUDE.md rule 8); splitting rules per book
live in `src/corpus.py`. Story title lines are dropped; chapter/section lines inside a story keep their
own paragraph numbers.

`write_story` refuses to overwrite an existing file, because annotations depend on paragraph numbers
(rule 2). Check the printed first/last paragraphs before running the last cell.

In [ ]:
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import pandas as pd
from src.corpus import BOOKS, download, book_title, book_stories, norm_title, write_story, PROCESSED

In [ ]:
for gid in BOOKS:
    path = download(gid)
    print(gid, path.name, "|", book_title(path))

In [ ]:
# Match each candidate (data/corpus_candidates.csv) to exactly one story heading
cand = pd.read_csv("data/corpus_candidates.csv")
found = {}
for gid in BOOKS:
    for title, paras in book_stories(gid):
        found.setdefault(norm_title(title), []).append((gid, title, paras))

rows = []
for _, r in cand.iterrows():
    hits = found.get(norm_title(r.title), [])
    assert len(hits) == 1, (r.list_id, r.title, [(g, t) for g, t, _ in hits])
    gid, title, paras = hits[0]
    rows.append({"story_id": r.list_id, "gutenberg_id": gid, "heading": title, "n_paragraphs": len(paras),
                 "first": paras[0][:60], "last": paras[-1][-60:], "exists": (PROCESSED / f"{r.list_id}.txt").exists()})
plan = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 70)
plan

In [ ]:
# ★ Check first/last above, then write the files that do not exist yet
for _, r in plan[~plan.exists].iterrows():
    paras = next(p for g, t, p in found[norm_title(r.heading)] if g == r.gutenberg_id)
    write_story(r.story_id, paras)
    print("wrote", r.story_id, r.n_paragraphs)